[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_15_Open_Source_Publishing.ipynb)

# 🌍 Lesson 15: Open-Source Project Publishing
### *Turning Your AutoResearcher Agent Into a Portfolio-Grade Open-Source Project*

---

**Course:** AI/LLM/Agents Engineering from Scratch  
**Lesson:** 15 of 15 (Phase 2 Complete!)  
**Prerequisites:** Lessons 1–14 (especially Lesson 9: AutoResearcher Capstone and Lesson 14: MCP)  
**Time:** ~2.5 hours  

---

## 🎓 What You'll Learn

You've built something real over the past 14 lessons — an AutoResearcher agent with:
- ReAct loop + tool use
- RAG with ChromaDB
- Multi-agent critic/reviewer pattern
- Structured outputs with Pydantic
- LangGraph stateful graphs
- MCP server interface
- Multimodal capabilities

**Today you learn how to ship it.** Open source isn't just putting code on GitHub — it's packaging your work so others can find it, understand it, install it, and trust it. That's the difference between a learning project and a career-changing portfolio piece.

By the end of this lesson you'll have:
1. A professional Python package structure
2. A `pyproject.toml` that makes your project installable via `pip`
3. A README that gets people to star and use your project
4. GitHub Actions CI/CD that runs your tests automatically
5. Community files (CONTRIBUTING, CODE_OF_CONDUCT, CHANGELOG)
6. Knowledge of how to publish to PyPI
7. A complete open-source AI agent project to put on your resume


---
## 🚀 Setup

In [ ]:
# Install required packages
!pip install anthropic pydantic chromadb pathlib2 -q

# Load API key from Colab Secrets
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Running locally
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"
    print("⚠️  Using environment variable or placeholder — set your key!")

import anthropic
import json
import os
from pathlib import Path
from textwrap import dedent

client = anthropic.Anthropic()
print("✅ Setup complete — let's ship your agent to the world!")

---
## 📖 Part 1: Why Open Source Matters For Your Career

Before we write a single config file, let's understand **why** this matters.

### The Open Source Signal

When you apply for an AI engineering role, a hiring manager looks at your GitHub profile. They don't want to see a graveyard of half-finished tutorials. They want to see:

- **A real problem solved** — not a hello-world demo
- **Professional structure** — shows you know engineering, not just Python scripting
- **Runnable code** — they can clone it and verify it works
- **Community engagement** — stars, forks, issues = social proof
- **Documentation** — shows communication skills, which senior engineers care about

### The AutoResearcher Is Your Proof

Over these 14 lessons, you built an agent that:
1. Takes a research question
2. Searches the web and reads papers
3. Stores knowledge in a vector database
4. Synthesizes findings with a critic reviewing quality
5. Outputs structured, validated research reports
6. Exposes itself as an MCP server

That's not a toy. That's an AI engineering portfolio piece. Let's package it properly.

---

## 📁 Part 2: Python Package Structure

There are two common layouts for Python packages:

### Flat Layout (simple, older)
```
auto_researcher/
├── auto_researcher/        ← your package
│   ├── __init__.py
│   ├── agent.py
│   └── tools.py
├── tests/
├── README.md
└── pyproject.toml
```

### src Layout (modern, recommended)
```
auto_researcher/
├── src/
│   └── auto_researcher/    ← your package
│       ├── __init__.py
│       ├── agent.py
│       ├── tools.py
│       ├── memory.py
│       └── models.py
├── tests/
│   ├── __init__.py
│   ├── test_agent.py
│   └── test_tools.py
├── docs/
├── examples/
│   └── research_demo.py
├── .github/
│   └── workflows/
│       └── ci.yml
├── README.md
├── CONTRIBUTING.md
├── CODE_OF_CONDUCT.md
├── CHANGELOG.md
├── LICENSE
└── pyproject.toml
```

**Why `src/` layout?** It prevents accidental imports of your development code instead of the installed package. Modern Python packaging tools (hatch, flit, poetry) all favor this.

Let's generate this entire structure:

In [ ]:
# Generate the full project structure in the current directory
# This simulates what you'd do locally when starting your OSS project

import os
from pathlib import Path

PROJECT_ROOT = Path("auto_researcher")

# All directories to create
dirs = [
    PROJECT_ROOT / "src" / "auto_researcher",
    PROJECT_ROOT / "tests",
    PROJECT_ROOT / "docs",
    PROJECT_ROOT / "examples",
    PROJECT_ROOT / ".github" / "workflows",
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"📁 Created: {d}")

print("\n✅ Project skeleton ready!")

# 💡 EXPERIMENT: Modify the structure above to add a 'benchmarks/' folder
# for performance testing, or a 'scripts/' folder for CLI utilities

---
## 📦 Part 3: pyproject.toml — Modern Python Packaging

`pyproject.toml` is the single configuration file that defines everything about your Python package: metadata, dependencies, build system, dev tools, test config.

Before it existed, projects had `setup.py`, `setup.cfg`, `MANIFEST.in`, `requirements.txt`, `tox.ini` — all separate files doing overlapping things. `pyproject.toml` consolidates all of it.

### Key sections:
- `[build-system]` — which tool builds your package (hatchling, setuptools, flit, poetry)
- `[project]` — name, version, description, dependencies, Python version requirement
- `[project.scripts]` — creates CLI commands when installed (`auto-researcher research "topic"`)
- `[project.optional-dependencies]` — extra deps for dev/testing
- `[tool.*]` — config for linters, formatters, test runners

In [ ]:
# Write pyproject.toml — the heart of a modern Python package

pyproject_toml = '''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "auto-researcher"
version = "0.1.0"
description = "An AI-powered research agent using Claude, RAG, and multi-agent patterns"
readme = "README.md"
license = { file = "LICENSE" }
authors = [
    { name = "Gourav Khanijoe", email = "gouravkhanijoe@gmail.com" }
]
keywords = ["ai", "llm", "agents", "rag", "research", "anthropic", "claude"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Intended Audience :: Developers",
    "License :: OSI Approved :: MIT License",
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3.10",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
    "Topic :: Scientific/Engineering :: Artificial Intelligence",
]
requires-python = ">=3.10"
dependencies = [
    "anthropic>=0.40.0",
    "pydantic>=2.0",
    "chromadb>=0.5.0",
    "httpx>=0.27.0",
    "rich>=13.0",           # Pretty terminal output
    "tenacity>=8.0",        # Retry logic
    "python-dotenv>=1.0",   # .env file support
]

# CLI command: users can run `auto-researcher research "What is RAG?"` after pip install
[project.scripts]
auto-researcher = "auto_researcher.cli:main"

# Optional dependencies: `pip install auto-researcher[dev]` installs test/dev tools
[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "pytest-asyncio>=0.23",
    "pytest-cov>=5.0",
    "ruff>=0.4",             # Extremely fast Python linter + formatter
    "mypy>=1.10",            # Static type checking
    "httpx>=0.27",
]
docs = [
    "mkdocs>=1.6",
    "mkdocs-material>=9.5",
    "mkdocstrings[python]>=0.25",
]

[project.urls]
Homepage = "https://github.com/gouravkhanijoe/auto-researcher"
Documentation = "https://gouravkhanijoe.github.io/auto-researcher"
Repository = "https://github.com/gouravkhanijoe/auto-researcher"
Issues = "https://github.com/gouravkhanijoe/auto-researcher/issues"
Changelog = "https://github.com/gouravkhanijoe/auto-researcher/blob/main/CHANGELOG.md"

# Hatch configuration: tells hatch where the source code lives
[tool.hatch.build.targets.wheel]
packages = ["src/auto_researcher"]

# Ruff: linter + formatter (replaces black + isort + flake8)
[tool.ruff]
line-length = 100
target-version = "py310"

[tool.ruff.lint]
select = ["E", "F", "I", "N", "W", "UP"]
ignore = ["E501"]  # Don't enforce line length in linter (formatter handles it)

# MyPy: type checking configuration
[tool.mypy]
python_version = "3.10"
warn_return_any = true
warn_unused_configs = true
ignore_missing_imports = true  # Some LLM libs don't have stubs yet

# Pytest configuration
[tool.pytest.ini_options]
asyncio_mode = "auto"
testpaths = ["tests"]
addopts = [
    "--tb=short",
    "--strict-markers",
]
markers = [
    "integration: marks tests as integration tests (require API key)",
    "unit: marks tests as unit tests (no external dependencies)",
]

# Coverage configuration
[tool.coverage.run]
source = ["src"]
omit = ["tests/*", "examples/*"]

[tool.coverage.report]
exclude_lines = [
    "pragma: no cover",
    "def __repr__",
    "raise NotImplementedError",
    "if TYPE_CHECKING:",
]
'''

(PROJECT_ROOT / "pyproject.toml").write_text(pyproject_toml)
print("✅ pyproject.toml written!")

# Let's also look at it
print("\n--- pyproject.toml (first 30 lines) ---")
lines = pyproject_toml.split("\n")
print("\n".join(lines[:30]))
print(f"... ({len(lines)} lines total)")

# 💡 EXPERIMENT: Notice the [project.scripts] section — after `pip install`,
# users can run `auto-researcher research "topic"` directly from their terminal.
# Try adding a second CLI command, e.g., `auto-researcher serve` for the MCP server.

---
## 📄 Part 4: The Package Source Code Structure

Let's build the actual `src/auto_researcher/` package. In a real project, this code would be your full AutoResearcher from Lesson 9. Here we create the proper structure with well-documented stubs.

### Key concept: `__init__.py` is your public API

When someone does `from auto_researcher import AutoResearcher`, Python looks in `__init__.py`. What you export there is your **public API contract** — the thing you promise not to break between versions.

```python
# Good: clean public API
from auto_researcher import AutoResearcher, ResearchReport

# Bad: internal details leaking out
from auto_researcher._internal._chromadb_store import _VectorStore  # messy
```

In [ ]:
# --- src/auto_researcher/__init__.py ---
# This is the public API of your package

init_py = '''"""
auto_researcher: An AI-powered research agent using Claude, RAG, and multi-agent patterns.

Basic usage:
    >>> from auto_researcher import AutoResearcher
    >>> agent = AutoResearcher(api_key="sk-ant-...")
    >>> report = agent.research("What are the latest advances in agentic AI systems?")
    >>> print(report.summary)
"""

from auto_researcher.agent import AutoResearcher
from auto_researcher.models import ResearchReport, Finding, Citation
from auto_researcher.exceptions import AutoResearcherError, RateLimitError, InvalidQueryError

__version__ = "0.1.0"
__author__ = "Gourav Khanijoe"
__email__ = "gouravkhanijoe@gmail.com"

# Public API — only these names are exported when someone does `from auto_researcher import *`
__all__ = [
    "AutoResearcher",
    "ResearchReport",
    "Finding",
    "Citation",
    "AutoResearcherError",
    "RateLimitError",
    "InvalidQueryError",
]
'''

# --- src/auto_researcher/models.py ---
# Pydantic models for structured outputs (from Lesson 10)
models_py = '''"""
Data models for the AutoResearcher agent.
All models use Pydantic v2 for validation and serialization.
"""

from __future__ import annotations
from datetime import datetime
from typing import Optional
from pydantic import BaseModel, Field, field_validator


class Citation(BaseModel):
    """A single source citation used in a research report."""

    url: str = Field(description="URL of the source")
    title: str = Field(description="Title of the source document")
    author: Optional[str] = Field(default=None, description="Author(s) if available")
    accessed_at: datetime = Field(default_factory=datetime.utcnow)
    relevance_score: float = Field(
        ge=0.0, le=1.0, description="Semantic similarity score from RAG retrieval"
    )


class Finding(BaseModel):
    """A single research finding with supporting evidence."""

    claim: str = Field(description="The finding or conclusion")
    evidence: list[str] = Field(
        default_factory=list, description="Supporting quotes or paraphrases"
    )
    citations: list[Citation] = Field(default_factory=list)
    confidence: float = Field(
        ge=0.0, le=1.0, description="Agent confidence in this finding (0-1)"
    )

    @field_validator("claim")
    @classmethod
    def claim_must_be_substantive(cls, v: str) -> str:
        if len(v.strip()) < 20:
            raise ValueError("Claim must be at least 20 characters")
        return v.strip()


class ResearchReport(BaseModel):
    """A complete research report generated by the AutoResearcher agent."""

    query: str = Field(description="The original research question")
    summary: str = Field(description="Executive summary of findings (2-3 paragraphs)")
    findings: list[Finding] = Field(
        default_factory=list, description="Key findings with evidence"
    )
    limitations: list[str] = Field(
        default_factory=list, description="Known limitations of this research"
    )
    generated_at: datetime = Field(default_factory=datetime.utcnow)
    model_used: str = Field(default="claude-sonnet-4-5")
    total_tokens_used: int = Field(default=0, ge=0)
    total_cost_usd: float = Field(default=0.0, ge=0.0)

    def to_markdown(self) -> str:
        """Render the report as a Markdown string for display or export."""
        lines = [
            f"# Research Report: {self.query}",
            f"*Generated: {self.generated_at.strftime(\\"%-d %b %Y %H:%M UTC\\")}*",
            f"*Model: {self.model_used} | Cost: ${self.total_cost_usd:.4f}*",
            "",
            "## Summary",
            self.summary,
            "",
            "## Key Findings",
        ]
        for i, finding in enumerate(self.findings, 1):
            lines.append(f"\\n### {i}. {finding.claim}")
            lines.append(f"*Confidence: {finding.confidence:.0%}*")
            if finding.evidence:
                lines.append("\\n**Evidence:**")
                for e in finding.evidence:
                    lines.append(f"- {e}")
        if self.limitations:
            lines.extend(["", "## Limitations"])
            for lim in self.limitations:
                lines.append(f"- {lim}")
        return "\\n".join(lines)
'''

# --- src/auto_researcher/exceptions.py ---
exceptions_py = '''"""
Custom exception hierarchy for auto_researcher.

Why custom exceptions? They let callers catch specific error types:
    try:
        report = agent.research(query)
    except RateLimitError:
        time.sleep(60); retry()
    except InvalidQueryError as e:
        print(f"Fix your query: {e}")
"""


class AutoResearcherError(Exception):
    """Base exception for all auto_researcher errors."""


class RateLimitError(AutoResearcherError):
    """Raised when the upstream LLM API rate limit is hit."""


class InvalidQueryError(AutoResearcherError):
    """Raised when the research query is malformed or too vague."""


class RAGError(AutoResearcherError):
    """Raised when the vector store or embedding pipeline fails."""


class CriticRejectionError(AutoResearcherError):
    """Raised when the critic agent rejects the report quality after max retries."""

    def __init__(self, message: str, quality_score: float, retries: int):
        self.quality_score = quality_score
        self.retries = retries
        super().__init__(message)
'''

# Write all source files
src_dir = PROJECT_ROOT / "src" / "auto_researcher"
(src_dir / "__init__.py").write_text(init_py)
(src_dir / "models.py").write_text(models_py)
(src_dir / "exceptions.py").write_text(exceptions_py)

# Create stub files for the main modules (agent.py, tools.py, memory.py, cli.py)
for stub_name, stub_desc in [
    ("agent.py", "AutoResearcher: main agent class with ReAct loop (from Lesson 9)"),
    ("tools.py", "Tool definitions: web_search, read_url, store_memory, recall_memory"),
    ("memory.py", "ChromaDB-backed vector store for RAG (from Lesson 7)"),
    ("cli.py", "CLI interface using argparse — `auto-researcher research <query>`"),
]:
    (src_dir / stub_name).write_text(
        f'"""\n{stub_desc}\n\nTODO: Integrate full implementation from Lesson 9 capstone.\n"""\n'
    )

print("✅ Package source structure created!")
print("\nFiles created:")
for f in sorted(src_dir.rglob("*")):
    print(f"  {f}")

# 💡 EXPERIMENT: In a real project, you'd copy your full Lesson 9 agent code into agent.py
# and tools.py. The models.py and exceptions.py above are already production-quality.

---
## 📝 Part 5: Writing a README That Gets Stars

The README is the most important file in your repository. It's the first thing people see, and it answers the three questions every visitor has:

1. **What does this do?** (in one sentence)
2. **Why should I care?** (problem + solution)
3. **Can I try it right now?** (quickstart in < 2 minutes)

### Anatomy of a great AI project README:

```
[Logo/Banner]
[Badges: PyPI version | Python | License | Tests | Coverage]

# Project Name — one-line description

## What it does (3-5 sentences with demo GIF/screenshot)

## Features (bullet list, 5-8 items max)

## Quick Start (copy-pasteable code, < 5 lines)

## Installation (pip install + any extras)

## Usage (3-4 realistic examples)

## Architecture (diagram showing agent flow)

## Contributing (link to CONTRIBUTING.md)

## License
```

Let's use Claude to help write a compelling README:

In [ ]:
# Use Claude to draft our README — a great example of AI-assisted development

import anthropic

client = anthropic.Anthropic()

readme_prompt = """
Write a compelling GitHub README.md for an open-source Python project called "auto-researcher".

Project facts:
- An AI agent that autonomously researches topics using Claude (Anthropic's LLM)
- Uses RAG (ChromaDB vector store) to ground findings in retrieved evidence
- Multi-agent architecture: researcher + critic reviewer pattern
- Structured outputs via Pydantic (ResearchReport model)
- Exposes as an MCP server (Model Context Protocol) for Claude Desktop integration
- CLI: `auto-researcher research "What is RAG?"`
- Python 3.10+, pip installable
- Built by a software engineer learning AI/LLM engineering

Instructions:
- Start with a 1-line hook description after the title
- Include PyPI, Python, and License badges (use placeholder URLs)
- Write a "What it does" section with a realistic code example
- Include a Features bullet list (max 8 items)
- Write a Quick Start section with pip install + a working example
- Include an Architecture section with a simple ASCII diagram showing the agent flow
- Keep total length to ~150 lines
- Tone: professional but approachable, aimed at developers
"""

print("Asking Claude to draft our README...")
response = client.messages.create(
    model="claude-haiku-4-5-20251001",  # Fast model for drafting
    max_tokens=2000,
    messages=[{"role": "user", "content": readme_prompt}]
)

readme_content = response.content[0].text

# Save it
(PROJECT_ROOT / "README.md").write_text(readme_content)

# Show the first 60 lines
lines = readme_content.split("\n")
print("\n" + "=" * 60)
print("GENERATED README (first 60 lines):")
print("=" * 60)
print("\n".join(lines[:60]))
print(f"\n... ({len(lines)} lines total, saved to README.md)")

# 💡 EXPERIMENT: Try asking Claude to make the README more specific:
# 'Rewrite the Quick Start section to show how to use it as an MCP server'
# Good READMEs are iteratively refined — don't just accept the first draft

---
## 🛡️ Part 6: License Selection

**Don't skip the license.** Without a license, your code is technically "all rights reserved" — no one can legally use it.

### The three licenses you'll see in AI projects:

| License | Who uses it | Key property |
|---------|------------|---------------|
| **MIT** | LangChain, most Anthropic examples | Do whatever you want, just keep the copyright notice |
| **Apache 2.0** | TensorFlow, Transformers (Hugging Face) | Like MIT but explicit patent grant — better for corporate use |
| **Apache 2.0 + Commons Clause** | Some commercial AI projects | Prevents others from selling your software as a service |

**For a portfolio project: use MIT.** It maximizes adoption and shows you're not worried about someone "stealing" it — your goal is visibility and contribution, not revenue.

Special case for AI models: some model licenses (Llama, Mistral) use custom licenses with usage restrictions. Always check before building a product on top of a model.

In [ ]:
# Write the MIT License
from datetime import datetime

mit_license = f"""MIT License

Copyright (c) {datetime.now().year} Gourav Khanijoe

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

(PROJECT_ROOT / "LICENSE").write_text(mit_license)
print("✅ MIT License written")
print(mit_license[:400])

# 💡 EXPERIMENT: Visit https://choosealicense.com — it's GitHub's official tool
# for comparing licenses. If you ever plan to build a business from this, 
# look into Business Source License (BSL) which converts to open source after N years.

---
## 🧪 Part 7: Writing Tests

Tests are what separate a script from production software. For AI agents, testing is tricky because LLM outputs are non-deterministic — but you can still test:

1. **Unit tests** — test individual functions with mocked LLM responses
2. **Integration tests** — test the full pipeline with a real API call (mark these separately)
3. **Contract tests** — verify your Pydantic models validate correctly
4. **Regression tests** — if a bug was found, write a test for it

### The key trick for testing LLM code: **dependency injection**

Instead of hardcoding `anthropic.Anthropic()` inside your agent, pass the client in as a parameter. Then tests can inject a mock client.

In [ ]:
# --- tests/test_models.py ---
# Testing the Pydantic models doesn't need an API key!

test_models_py = '''"""
Tests for data models — no API key required.
Run: pytest tests/test_models.py -v
"""

import pytest
from datetime import datetime

# Note: in a real project, these would import from your installed package
# from auto_researcher import ResearchReport, Finding, Citation
# For this demo, we inline simplified versions

from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional


class Citation(BaseModel):
    url: str
    title: str
    relevance_score: float = Field(ge=0.0, le=1.0)


class Finding(BaseModel):
    claim: str
    confidence: float = Field(ge=0.0, le=1.0)

    @field_validator("claim")
    @classmethod
    def claim_must_be_substantive(cls, v: str) -> str:
        if len(v.strip()) < 20:
            raise ValueError("Claim must be at least 20 characters")
        return v.strip()


# --- Tests ---

class TestCitation:
    def test_valid_citation(self):
        c = Citation(
            url="https://example.com",
            title="Test Paper",
            relevance_score=0.92
        )
        assert c.relevance_score == 0.92

    def test_relevance_score_out_of_range(self):
        with pytest.raises(ValidationError) as exc_info:
            Citation(url="https://x.com", title="T", relevance_score=1.5)
        assert "relevance_score" in str(exc_info.value)

    def test_relevance_score_negative(self):
        with pytest.raises(ValidationError):
            Citation(url="https://x.com", title="T", relevance_score=-0.1)


class TestFinding:
    def test_valid_finding(self):
        f = Finding(claim="RAG significantly improves factual accuracy", confidence=0.85)
        assert f.confidence == 0.85

    def test_claim_too_short(self):
        with pytest.raises(ValidationError) as exc_info:
            Finding(claim="Too short", confidence=0.5)
        assert "20 characters" in str(exc_info.value)

    def test_claim_gets_stripped(self):
        f = Finding(
            claim="  RAG improves accuracy of large language models significantly  ",
            confidence=0.9
        )
        assert not f.claim.startswith(" ")
        assert not f.claim.endswith(" ")
'''

# --- tests/conftest.py ---
# Fixtures shared across all tests
conftest_py = '''"""
Pytest fixtures and configuration shared across all tests.
"""

import pytest
import os


@pytest.fixture(scope="session")
def api_key():
    """Return the Anthropic API key, skip integration tests if not set."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        pytest.skip("ANTHROPIC_API_KEY not set — skipping integration test")
    return key


@pytest.fixture
def sample_finding():
    """A pre-built Finding for use in multiple tests."""
    return {
        "claim": "Retrieval augmented generation reduces hallucination significantly",
        "confidence": 0.88,
    }
'''

(PROJECT_ROOT / "tests" / "__init__.py").write_text("")
(PROJECT_ROOT / "tests" / "test_models.py").write_text(test_models_py)
(PROJECT_ROOT / "tests" / "conftest.py").write_text(conftest_py)

print("✅ Test files written")

# Now let's actually RUN the tests to verify they pass
print("\n--- Running tests ---")
!pip install pytest pydantic -q
!cd auto_researcher && python -m pytest tests/test_models.py -v 2>&1 | head -40

# 💡 EXPERIMENT: Add a test that verifies a Finding with confidence=1.1 raises ValidationError
# Then run pytest again to see the new test pass

---
## ⚙️ Part 8: GitHub Actions — Automated CI/CD

**CI/CD** = Continuous Integration / Continuous Delivery. Every time you push code to GitHub, a robot automatically:
1. Checks out your code
2. Sets up Python
3. Installs dependencies
4. Runs your linter
5. Runs your tests
6. Reports pass/fail with a badge on your README

This badge → `[![Tests](https://img.shields.io/github/actions/workflow/status/you/repo/ci.yml)](...)` signals to every visitor: "this code works."

GitHub Actions is free for public repositories (unlimited minutes).

In [ ]:
# --- .github/workflows/ci.yml ---
# This runs on every push and pull request to main/develop branches

ci_yml = '''name: CI

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

jobs:
  test:
    name: Test on Python ${{ matrix.python-version }}
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip  # Cache pip downloads for faster runs

      - name: Install dependencies
        run: |
          pip install -e ".[dev]"
          # -e installs in editable mode so tests import from src/

      - name: Lint with ruff
        run: |
          ruff check src/ tests/
          ruff format --check src/ tests/  # Fail if formatting differs

      - name: Type check with mypy
        run: mypy src/auto_researcher/
        continue-on-error: true  # Don\'t fail CI on type errors yet (we\'re learning)

      - name: Run unit tests
        run: |
          pytest tests/ -m "not integration" -v --cov=src/auto_researcher --cov-report=xml
          # -m "not integration" skips tests that need an API key

      - name: Upload coverage report
        uses: codecov/codecov-action@v4
        with:
          token: ${{ secrets.CODECOV_TOKEN }}
          file: ./coverage.xml
          fail_ci_if_error: false

  # Only run integration tests on the main branch (needs API key secret)
  integration-test:
    name: Integration Tests
    runs-on: ubuntu-latest
    if: github.ref == \'refs/heads/main\'
    needs: test  # Only run if unit tests pass first

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install
        run: pip install -e ".[dev]"

      - name: Run integration tests
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
          # Add this secret in GitHub: Settings → Secrets → Actions → New repo secret
        run: pytest tests/ -m integration -v
'''

(PROJECT_ROOT / ".github" / "workflows" / "ci.yml").write_text(ci_yml)
print("✅ GitHub Actions CI workflow written")

# --- Also create a release workflow ---
release_yml = '''name: Release to PyPI

# Trigger this workflow when you push a version tag: git tag v0.1.0 && git push --tags
on:
  push:
    tags:
      - "v*"  # Matches v0.1.0, v1.2.3, etc.

jobs:
  release:
    name: Build and publish to PyPI
    runs-on: ubuntu-latest
    environment: pypi  # Requires manual approval in GitHub Environments settings

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install build tools
        run: pip install hatch

      - name: Build distribution
        run: hatch build
        # Creates dist/*.whl and dist/*.tar.gz

      - name: Publish to PyPI
        uses: pypa/gh-action-pypi-publish@release/v1
        with:
          password: ${{ secrets.PYPI_API_TOKEN }}
          # Add this in: repo Settings → Secrets → PYPI_API_TOKEN
'''

(PROJECT_ROOT / ".github" / "workflows" / "release.yml").write_text(release_yml)
print("✅ Release workflow written")

print("\n📋 GitHub Actions summary:")
print("  ci.yml     — runs on every push: lint + type-check + unit tests")
print("  release.yml — runs on version tag push: builds + publishes to PyPI")

# 💡 EXPERIMENT: Look at the 'matrix' strategy in ci.yml — it runs your tests
# on Python 3.10, 3.11, AND 3.12 in parallel. This catches version-specific bugs.
# Try removing 3.10 from the matrix and see what changes conceptually.

---
## 🤝 Part 9: Community Files

These files signal that your project is welcoming and maintained:

- **CONTRIBUTING.md** — how to submit changes (setup, PR process, code style)
- **CODE_OF_CONDUCT.md** — community behavior expectations (use the standard Contributor Covenant)
- **CHANGELOG.md** — version history (what changed in each release)
- **SECURITY.md** — how to report security vulnerabilities (don't open a public issue!)

GitHub shows special badges for projects that have these files. The Contributor Covenant is the standard used by Linux, Node.js, Angular, etc.

In [ ]:
# Write community files

contributing_md = """# Contributing to auto-researcher

First off, thank you for considering contributing! 🎉

## Development Setup

```bash
# Clone the repo
git clone https://github.com/gouravkhanijoe/auto-researcher
cd auto-researcher

# Create a virtual environment
python -m venv .venv
source .venv/bin/activate  # On Windows: .venv\\Scripts\\activate

# Install in editable mode with dev dependencies
pip install -e ".[dev]"

# Set your API key
export ANTHROPIC_API_KEY="sk-ant-..."

# Run tests to verify setup
pytest tests/ -m "not integration" -v
```

## How to Contribute

1. **Open an issue first** — discuss the change before writing code
2. **Fork the repo** and create a feature branch: `git checkout -b feature/your-feature-name`
3. **Write your code** following the style guide (ruff handles formatting)
4. **Write tests** — we aim for >80% coverage on new code
5. **Open a PR** against the `main` branch with a clear description

## Code Style

We use `ruff` for linting and formatting. Run before committing:
```bash
ruff format src/ tests/
ruff check src/ tests/ --fix
```

## Commit Message Format

We follow [Conventional Commits](https://www.conventionalcommits.org/):
```
feat: add streaming support to research() method
fix: handle empty search results gracefully
docs: add example for custom tool integration
test: add integration test for critic agent
chore: bump anthropic to 0.45.0
```

## Questions?

Open a [Discussion](https://github.com/gouravkhanijoe/auto-researcher/discussions) — not an Issue.
Issues are for bugs and feature requests, Discussions are for questions.
"""

changelog_md = """# Changelog

All notable changes to this project will be documented in this file.

The format follows [Keep a Changelog](https://keepachangelog.com/en/1.0.0/).
This project adheres to [Semantic Versioning](https://semver.org/spec/v2.0.0.html).

## [Unreleased]

### Planned
- Async support for parallel research tasks
- Web UI via Streamlit
- Support for PDF document ingestion
- GPT-4o and Gemini model backends

---

## [0.1.0] - 2026-05-14

### Added
- Initial release 🎉
- `AutoResearcher` agent with ReAct loop
- RAG pipeline using ChromaDB
- Multi-agent critic/reviewer pattern
- Structured `ResearchReport` output with Pydantic
- MCP server interface for Claude Desktop
- CLI: `auto-researcher research <query>`
- GitHub Actions CI/CD pipeline
- Comprehensive test suite
"""

security_md = """# Security Policy

## Reporting a Vulnerability

**Do NOT open a public GitHub issue for security vulnerabilities.**

Instead, please email: gouravkhanijoe@gmail.com

Include:
- Description of the vulnerability
- Steps to reproduce
- Potential impact
- Suggested fix (if any)

I will acknowledge receipt within 48 hours and aim to address critical issues within 7 days.

## Scope

- Prompt injection vulnerabilities in the agent's tool execution
- Data leakage between research sessions
- API key exposure in logs or error messages
"""

for filename, content in [
    ("CONTRIBUTING.md", contributing_md),
    ("CHANGELOG.md", changelog_md),
    ("SECURITY.md", security_md),
]:
    (PROJECT_ROOT / filename).write_text(content)
    print(f"✅ {filename} written")

print("\n💡 For CODE_OF_CONDUCT.md, visit https://www.contributor-covenant.org/")
print("   Copy the boilerplate — it's used by 40,000+ open source projects")

# 💡 EXPERIMENT: Read the CHANGELOG format above — notice how it uses
# semantic versioning (MAJOR.MINOR.PATCH). When should you bump each?
# - PATCH (0.1.1): Bug fix, backward compatible
# - MINOR (0.2.0): New feature, backward compatible
# - MAJOR (1.0.0): Breaking change

---
## 🚀 Part 10: Publishing to PyPI

PyPI (the Python Package Index) is where `pip install X` downloads from. Publishing to PyPI means anyone in the world can install your package with one command.

### The process:

```
1. Create account at pypi.org (and test.pypi.org for testing)
2. Generate an API token (Settings → API Tokens)
3. Build your package:
   pip install hatch
   hatch build
   # → dist/auto_researcher-0.1.0-py3-none-any.whl
   # → dist/auto_researcher-0.1.0.tar.gz

4. Upload to TestPyPI first (safe to experiment):
   pip install twine
   twine upload --repository testpypi dist/*

5. Install from TestPyPI to verify it works:
   pip install --index-url https://test.pypi.org/simple/ auto-researcher

6. Upload to real PyPI:
   twine upload dist/*
```

After step 6, anyone can do: `pip install auto-researcher` ✨

**With the Release GitHub Action we wrote**, steps 3-6 happen automatically whenever you push a version tag:
```bash
git tag v0.1.0
git push origin v0.1.0
# → GitHub Action builds + uploads to PyPI automatically
```

In [ ]:
# Let's actually BUILD the package to see what it produces
# (We won't publish to PyPI since this is a demo, but you can see the artifacts)

!pip install hatch -q

import subprocess
import os

# Create minimal __init__.py stubs so hatch can build without errors
# (In a real project, this would be your full code)
agent_stub = '''"""
AutoResearcher agent — main class.
See Lesson 9 for the full implementation.
"""

class AutoResearcher:
    """AI-powered research agent using Claude and RAG."""
    
    def __init__(self, api_key: str | None = None):
        import os
        self.api_key = api_key or os.environ.get("ANTHROPIC_API_KEY", "")
        if not self.api_key:
            raise ValueError("ANTHROPIC_API_KEY not set")
    
    def research(self, query: str) -> dict:
        """Run a research task and return a ResearchReport."""
        raise NotImplementedError("Integrate full Lesson 9 implementation here")
'''

cli_stub = '''"""
CLI interface for auto-researcher.
Usage: auto-researcher research "What is RAG?"
"""

import argparse
import sys

def main():
    parser = argparse.ArgumentParser(
        description="auto-researcher: AI-powered research agent"
    )
    subparsers = parser.add_subparsers(dest="command")
    
    research_cmd = subparsers.add_parser("research", help="Run a research task")
    research_cmd.add_argument("query", help="Research question")
    research_cmd.add_argument("--model", default="claude-sonnet-4-5", help="Claude model")
    research_cmd.add_argument("--output", default="report.md", help="Output file")
    
    args = parser.parse_args()
    
    if args.command == "research":
        print(f"Researching: {args.query}")
        print("(Full implementation: see Lesson 9)")
    else:
        parser.print_help()
        sys.exit(1)

if __name__ == "__main__":
    main()
'''

src_dir = Path("auto_researcher") / "src" / "auto_researcher"
(src_dir / "agent.py").write_text(agent_stub)
(src_dir / "cli.py").write_text(cli_stub)

# Build the package
result = subprocess.run(
    ["hatch", "build"],
    cwd="auto_researcher",
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Package built successfully!")
    print(result.stdout)
    # List what was produced
    dist_files = list(Path("auto_researcher/dist").glob("*"))
    print("\n📦 Distribution artifacts:")
    for f in dist_files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name} ({size_kb:.1f} KB)")
    print("\nTo publish: twine upload auto_researcher/dist/*")
else:
    print("Build output:")
    print(result.stdout[-1000:] if result.stdout else "(no stdout)")
    print(result.stderr[-1000:] if result.stderr else "(no stderr)")

# 💡 EXPERIMENT: Install the .whl file we just built:
#   !pip install auto_researcher/dist/*.whl
# Then try: from auto_researcher import AutoResearcher

---
## 📊 Part 11: GitHub Repository Best Practices

Getting the code up is step 1. Making the repo discoverable and collaborative takes a few more steps:

### Topics and Description
On your GitHub repo page, add **topics** (tags) — GitHub uses these for search:
```
ai-agent, llm, rag, anthropic, claude, python, multi-agent, mcp
```

### Branch Protection
Protect `main` so no one (including you) can push directly:
- Settings → Branches → Add rule for `main`
- ✅ Require pull request before merging
- ✅ Require status checks to pass (CI)
- ✅ Require linear history (cleaner git log)

### Issue Templates
When people open issues, guide them to give you useful information:

In [ ]:
# Create GitHub issue templates

github_dir = PROJECT_ROOT / ".github"
issue_templates_dir = github_dir / "ISSUE_TEMPLATE"
issue_templates_dir.mkdir(exist_ok=True)

bug_template = """---
name: Bug Report
about: Something isn't working
title: '[BUG] '
labels: bug
assignees: ''
---

## Describe the Bug
A clear description of what the bug is.

## Steps to Reproduce
```python
from auto_researcher import AutoResearcher
# minimal code that reproduces the issue
```

## Expected Behavior
What you expected to happen.

## Actual Behavior
What actually happened (include full error traceback).

## Environment
- auto-researcher version: (run `pip show auto-researcher`)
- Python version: 
- OS:
"""

feature_template = """---
name: Feature Request
about: Suggest a new capability
title: '[FEAT] '
labels: enhancement
assignees: ''
---

## Problem
What problem does this feature solve? (e.g., "I can't research topics that require image analysis because...")

## Proposed Solution
What would the API/behavior look like?
```python
# Example of how you'd like it to work
agent.research("topic", with_images=True)
```

## Alternatives Considered
Any workarounds you've tried?

## Additional Context
Any other relevant information.
"""

pr_template = """## Summary
What does this PR do? (one sentence)

## Changes
- Added/modified/removed X
- Added/modified/removed Y

## Testing
- [ ] Unit tests pass (`pytest tests/ -m "not integration"`)
- [ ] Integration tests pass (if applicable)
- [ ] Ruff linting passes (`ruff check src/ tests/`)

## Related Issue
Closes #(issue number)

## Breaking Changes
List any breaking API changes (or "None")
"""

(issue_templates_dir / "bug_report.md").write_text(bug_template)
(issue_templates_dir / "feature_request.md").write_text(feature_template)
(github_dir / "pull_request_template.md").write_text(pr_template)

print("✅ Issue and PR templates written")
print("\nWhen contributors open an issue on GitHub, they'll see a choice:")
print("  🐛 Bug Report")
print("  ✨ Feature Request")
print("\nThis dramatically improves issue quality (no more 'it broke' with zero detail)")

# 💡 EXPERIMENT: Go look at the issue templates for a popular AI project:
# https://github.com/langchain-ai/langchain/tree/master/.github/ISSUE_TEMPLATE
# Notice how they ask for reproduction steps and environment details upfront

---
## 🗺️ Part 12: Capstone — Review Your Complete Project

Let's take stock of everything we've built and generate a summary of the full project structure.

In [ ]:
# Print the complete project tree
from pathlib import Path

def print_tree(directory, indent=0, max_depth=5):
    """Print a directory tree."""
    if indent > max_depth:
        return
    
    d = Path(directory)
    items = sorted(d.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    
    for item in items:
        if item.name.startswith('__pycache__'):
            continue
        prefix = "  " * indent
        if item.is_dir():
            print(f"{prefix}📁 {item.name}/")
            print_tree(item, indent + 1, max_depth)
        else:
            size = item.stat().st_size
            size_str = f"{size:,}b" if size < 1024 else f"{size//1024}KB"
            print(f"{prefix}📄 {item.name} ({size_str})")

print("🌳 Complete auto-researcher project structure:")
print("="*55)
print_tree("auto_researcher")
print("="*55)

# Count files
all_files = list(Path("auto_researcher").rglob("*"))
py_files = [f for f in all_files if f.suffix == '.py']
md_files = [f for f in all_files if f.suffix == '.md']
config_files = [f for f in all_files if f.suffix in ('.toml', '.yml')]

print(f"\n📊 Project stats:")
print(f"  Python files:  {len(py_files)}")
print(f"  Markdown docs: {len(md_files)}")
print(f"  Config files:  {len(config_files)}")
print(f"  Total files:   {len([f for f in all_files if f.is_file()])}")

In [ ]:
# Generate a checklist: is this project ready to publish?
import anthropic

client = anthropic.Anthropic()

# Build a summary of what files exist
files_list = []
for f in sorted(Path("auto_researcher").rglob("*")):
    if f.is_file() and "__pycache__" not in str(f) and "dist" not in str(f):
        files_list.append(str(f.relative_to("auto_researcher")))

files_summary = "\n".join(files_list)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=800,
    messages=[{
        "role": "user",
        "content": f"""Review this Python open-source project structure and give a short assessment:
        
Files present:
{files_summary}

Give:
1. A 'Ready to publish' score out of 10
2. Top 3 things missing or to improve
3. One genuine strength
Keep it concise — 150 words max."""
    }]
)

print("🤖 Claude's assessment of your project readiness:")
print("="*55)
print(response.content[0].text)

# 💡 EXPERIMENT: Ask Claude what the #1 thing would be to make this project
# get 100 GitHub stars in the first month. The answer is usually about
# discoverability (README quality, social posting) not code quality.

---
## 📋 Part 13: Your Launch Checklist

Copy this to your notes — these are the exact steps to go from "code on my machine" to "open source project people can discover and use":

In [ ]:
# Print the full launch checklist

checklist = """
╔══════════════════════════════════════════════════════════════╗
║          🚀 OPEN SOURCE LAUNCH CHECKLIST                    ║
║          (for auto-researcher or any AI agent project)      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  📦 PACKAGE STRUCTURE                                       ║
║  [ ] src/ layout with __init__.py                           ║
║  [ ] pyproject.toml with metadata + deps                    ║
║  [ ] CLI entry point (project.scripts)                      ║
║  [ ] Custom exceptions hierarchy                            ║
║  [ ] Pydantic models for all inputs/outputs                 ║
║                                                              ║
║  📝 DOCUMENTATION                                           ║
║  [ ] README with demo, features, quickstart, architecture   ║
║  [ ] CONTRIBUTING.md                                        ║
║  [ ] CHANGELOG.md (v0.1.0 initial release)                  ║
║  [ ] SECURITY.md                                            ║
║  [ ] CODE_OF_CONDUCT.md (use Contributor Covenant)          ║
║  [ ] LICENSE (MIT for portfolio projects)                   ║
║  [ ] Docstrings on all public functions/classes             ║
║                                                              ║
║  🧪 QUALITY                                                 ║
║  [ ] Unit tests for models and utilities                    ║
║  [ ] Integration tests (marked separately)                  ║
║  [ ] Ruff linting passes                                    ║
║  [ ] .env.example file (never commit real keys!)            ║
║  [ ] .gitignore (Python template from gitignore.io)         ║
║                                                              ║
║  ⚙️  CI/CD                                                  ║
║  [ ] GitHub Actions: lint + test on every push              ║
║  [ ] GitHub Actions: release to PyPI on tag push            ║
║  [ ] ANTHROPIC_API_KEY secret added in repo settings        ║
║                                                              ║
║  🌐 GITHUB SETUP                                            ║
║  [ ] Repository description filled in                       ║
║  [ ] Topics/tags set (ai-agent, llm, rag, claude, etc.)     ║
║  [ ] Branch protection on main                              ║
║  [ ] Issue templates (bug report, feature request)          ║
║  [ ] PR template                                            ║
║  [ ] Discussions enabled (for community Q&A)               ║
║                                                              ║
║  📣 LAUNCH                                                  ║
║  [ ] Post on X/Twitter with demo GIF                        ║
║  [ ] Post on LinkedIn                                       ║
║  [ ] Submit to Hacker News (Show HN: ...)                   ║
║  [ ] Share in relevant Discord/Slack communities            ║
║  [ ] Post to r/MachineLearning or r/LocalLLaMA              ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""

print(checklist)

---
## 🏁 Phase 2 Complete — What You've Built

**Congratulations, Gourav.** You just completed both phases of the AI Engineering curriculum.

Here's what you can now put on your resume and GitHub:

### Phase 1 Skills Demonstrated:
- LLM API usage (streaming, token counting, error handling)
- Prompt engineering (chain-of-thought, few-shot, XML structuring)
- Tool use / function calling
- ReAct agent loop from scratch
- Agent memory (context window management, ChromaDB persistence)
- Multi-agent systems (orchestrator + subagent patterns)
- RAG pipeline (embeddings, vector search, chunking)
- Production AI (observability, evals, cost optimization)

### Phase 2 Skills Demonstrated:
- Structured outputs with Pydantic + instructor + auto-retry
- Stateful agent graphs with LangGraph
- Fine-tuning concepts (LoRA, PEFT, SFTTrainer)
- Multimodal AI (vision+text, chart analysis, RAG+vision)
- MCP server development (FastMCP, tools/resources/prompts)
- **Open-source publishing** (this lesson)

### Your Open-Source Portfolio:
**AutoResearcher** — an AI agent that:
- Takes a research question
- Uses a ReAct loop with web search tools
- Grounds findings in a ChromaDB vector store
- Validates outputs with Pydantic models
- Reviews quality with a critic subagent
- Exposes itself as an MCP server
- Installable via `pip install auto-researcher`

---

## 🔭 What's Coming in Phase 3

The learning doesn't stop. Phase 3 will take you from engineer to **expert**:

| # | Topic | Why it matters |
|---|-------|----------------|
| 16 | **Deployment** — FastAPI + Docker + Fly.io | Your agent as a real web service |
| 17 | **Advanced Evals** — RAGAS, LLM-judge pipelines | Measuring agent quality at scale |
| 18 | **AI Security** — Prompt injection, guardrails, red-teaming | Production safety |
| 19 | **Streaming Agents** — Server-sent events, real-time UIs | User-facing agent products |
| 20 | **Vector DB in Production** — Pinecone, Weaviate at scale | Beyond ChromaDB |
| 21 | **Agent Frameworks Deep Dive** — CrewAI, AutoGen, comparison | Industry tools |
| 22 | **Cost Engineering** — Model routing, caching, batching | Build economical agents |
| 23 | **Phase 3 Capstone** — Ship v1.0 of AutoResearcher | Your open-source portfolio piece |

See you in Lesson 16 🚀

In [ ]:
# 🎓 Final summary: print what you accomplished in this lesson

summary = """
╔══════════════════════════════════════════════════════════════╗
║           ✅ LESSON 15 COMPLETE                              ║
║           Open-Source Project Publishing                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Files you created:                                         ║
║  📦 pyproject.toml         — modern Python packaging        ║
║  🗂  src/auto_researcher/  — proper package structure       ║
║  🔍 models.py              — production Pydantic models     ║
║  ⚠️  exceptions.py         — custom exception hierarchy     ║
║  🧪 tests/test_models.py   — passing unit tests            ║
║  📝 README.md              — AI-assisted compelling README  ║
║  📜 LICENSE                — MIT license                   ║
║  🤝 CONTRIBUTING.md        — contribution guide            ║
║  📋 CHANGELOG.md           — version history               ║
║  🔒 SECURITY.md            — vulnerability reporting       ║
║  ⚙️  .github/workflows/ci.yml    — automated CI/CD         ║
║  🚀 .github/workflows/release.yml — auto PyPI publishing   ║
║  🐛 Issue + PR templates                                    ║
║  📦 dist/*.whl             — built, installable package    ║
║                                                              ║
║  Concepts you now own:                                      ║
║  ✅ src/ layout vs flat layout                              ║
║  ✅ pyproject.toml anatomy                                  ║
║  ✅ Semantic versioning (MAJOR.MINOR.PATCH)                 ║
║  ✅ Unit vs integration test separation                     ║
║  ✅ GitHub Actions CI matrix strategy                       ║
║  ✅ MIT vs Apache 2.0 license selection                     ║
║  ✅ PyPI publication flow                                   ║
║  ✅ Community file standards                                ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝

🎯 Next step: Take your Lesson 9 AutoResearcher code and
   integrate it into the package structure you built today.
   Then push to GitHub and share the link!

   Phase 3 starts tomorrow — we're going to deployment! 🌐
"""

print(summary)